In [1]:
import torch
import torchvision
from torch import nn, optim
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt
from torchvision import transforms, datasets
from GhostNet import GhostNet
from GhostNet_2 import GhostNet as ghost_change
from ghost_simple import SimpleGhost

In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果使用多GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 100
# 在训练代码开头调用
set_seed(seed) # 42, 2025, 1024, 512, 100

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307, ), (0.3081, )),
    transforms.Grayscale(1)
])

In [5]:
test_db = datasets.ImageFolder(root= 'C:/Users/Administrator/Desktop/datas/test_data_black', transform= transform)
test_loader = DataLoader(test_db, batch_size= 10, shuffle= True)
len(test_loader)

297

In [6]:
ghost = GhostNet(in_ch= 1)
ghost.load_state_dict(torch.load(f'models/complete/ghostnet_md_{seed}.pth'))
ghost.to(device)
ghost.eval()
print('ghostnet load')

ghostnet load


In [7]:
ghost_c = ghost_change(in_ch= 1)
ghost_c.load_state_dict(torch.load(f'models/complete/ghostnet_change_md_{seed}.pth'))
ghost_c.to(device)
ghost_c.eval()
print('ghostnet change load')

ghostnet change load


In [8]:
ghost_s = SimpleGhost(in_channels= 1)
ghost_s.load_state_dict(torch.load(f'models/simple/ghostnet_simple_md_{seed}.pth'))
ghost_s.to(device)
ghost_s.eval()
print('ghostnet change load')

ghostnet change load


In [9]:
total = 0
idxs = []
ghost_correct, ghost_c_correct, ghost_s_correct = 0, 0, 0
ghost_c_corrects, ghost_s_corrects, ghost_corrects = [], [], []
with torch.no_grad():
    for idx, (x, y) in enumerate(test_loader):
        x, y = x.to(device), y.to(device)
        out_ghost = torch.argmax(ghost(x), dim= -1)
        out_ghost_c = torch.argmax(ghost_c(x), dim= -1)
        out_ghost_s = torch.argmax(ghost_s(x), dim= -1)

        total += x.size(0)
        ghost_correct += (out_ghost == y).sum().item()
        ghost_c_correct += (out_ghost_c == y).sum().item()
        ghost_s_correct += (out_ghost_s == y).sum().item()

        if idx % 50 == 0 or (idx + 1) == len(test_loader):
            accu1 = ghost_correct / total
            accu2 = ghost_c_correct / total
            accu3 = ghost_s_correct / total

            idxs.append(idx)
            ghost_corrects.append(accu1)
            ghost_c_corrects.append(accu2)
            ghost_s_corrects.append(accu3)

            print(f'idx: {idx}, ghost:{accu1:.4f}, change:{accu2:.4f}, simple:{accu3:.4f}')

idx: 0, ghost:1.0000, change:1.0000, simple:1.0000
idx: 50, ghost:0.8961, change:0.9275, simple:0.9098
idx: 100, ghost:0.8861, change:0.9228, simple:0.9109
idx: 150, ghost:0.8894, change:0.9252, simple:0.9073
idx: 200, ghost:0.8925, change:0.9274, simple:0.9159
idx: 250, ghost:0.8936, change:0.9247, simple:0.9159
idx: 296, ghost:0.8909, change:0.9236, simple:0.9148


In [10]:
with open('data_csv/ghostnet_data.csv', '+a') as f:
    for idx, m1, m2, m3 in zip(idxs, ghost_corrects, ghost_c_corrects, ghost_s_corrects):
        f.writelines(f'{idx},{seed},{m1},{m2},{m3}\n')